# 04 · Batch effect: PCA before, ComBat-seq, PCA after

**Run order:** 02 make facility CSVs → 03 select samples + metadata → **04 (this notebook)**. Kernel: **R (WGCNA)**.

**What this does.** Takes the *eligible* samples from 03 and asks: do samples group by sequencing facility (`batch`) rather than by diagnosis? It draws a PCA (a 2-D map where similar samples sit close together) coloured by batch and by diagnosis, applies **ComBat-seq** to remove the batch shift while protecting diagnosis, and draws the PCA again. It prints how much of PC1/PC2 is explained by batch and by diagnosis before and after, so you don't have to judge it by eye.

**The corrected counts are for display (heatmaps, PCA).** Differential expression should use the **raw** counts with `batch` in the model and `patient_key` as a block.

**Reads:** `rnaseq_out_v2/rnaseq_counts_raw_eligible.csv`, `rnaseq_samples_eligible.csv` (from 03). **Writes (all CSV):** `rnaseq_out_v2/batch_correction/`
- `counts_combatseq_eligible.csv`: corrected counts, all eligible samples
- `counts_combatseq_one_per_patient.csv`: same, one sample per patient
- `pca_before.csv`, `pca_after.csv`: PC coordinates per sample
- `variance_explained_by_batch_and_diagnosis.csv`

In [ ]:
RESULTS <- "/home/jovyan/session_data/endotype_results"
IN_DIR  <- file.path(RESULTS, "rnaseq_out_v2")
OUT_DIR <- file.path(IN_DIR, "batch_correction")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)
MIN_BATCH <- 3     # batches with fewer samples than this are pooled into one group for ComBat-seq

need <- c("edgeR", "sva")
absent <- need[!vapply(need, requireNamespace, logical(1), quietly = TRUE)]
if (length(absent)) {                       # one-time install if the R environment lacks them
  if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager", repos = "https://cloud.r-project.org")
  BiocManager::install(absent, ask = FALSE, update = FALSE)
}
suppressMessages({library(edgeR); library(sva)})

## Read the eligible samples
Counts (genes × samples) and the sample table from notebook 03. The sample table is put in the same order as the count columns, and Ensembl versions are removed (`ENSG00000000003.10` → `ENSG00000000003`).

In [ ]:
cd <- read.csv(file.path(IN_DIR, "rnaseq_counts_raw_eligible.csv"), check.names = FALSE, row.names = 1)
gene_name <- cd$Gene_Name
counts <- as.matrix(cd[, colnames(cd) != "Gene_Name"]); storage.mode(counts) <- "numeric"
samples <- read.csv(file.path(IN_DIR, "rnaseq_samples_eligible.csv"), check.names = FALSE, colClasses = "character")
stopifnot(!anyNA(match(colnames(counts), samples$sample_id)))
samples <- samples[match(colnames(counts), samples$sample_id), ]
samples$diagnosis <- ifelse(samples$dx_clean == "", "(blank)", samples$dx_clean)

ens <- sub("\\.[0-9]+$", "", rownames(counts))
stopifnot(!anyDuplicated(ens), !anyNA(counts), all(counts >= 0), all(counts == round(counts)))
rownames(counts) <- ens
par_y <- startsWith(ens, "ENSGR")                       # pseudoautosomal copies; keep only if they have counts
keep <- !par_y | rowSums(counts) > 0
counts <- counts[keep, ]; gene_name <- gene_name[keep]
c(genes = nrow(counts), samples = ncol(counts), patients = length(unique(samples$patient_key)))

## Batch against diagnosis
A batch that holds a single diagnosis cannot be separated from it. ComBat-seq needs a few samples per batch, so batches smaller than `MIN_BATCH` are pooled into one group (`batch_cs`) for the correction only.

In [ ]:
table(batch = samples$batch, diagnosis = samples$diagnosis)
bsize <- table(samples$batch)
small <- names(bsize)[bsize < MIN_BATCH]
samples$batch_cs <- ifelse(samples$batch %in% small, "pooled-small-batches", samples$batch)
if (length(small)) cat("pooled for ComBat-seq (fewer than", MIN_BATCH, "samples):", small, "\n")
if (any(table(samples$batch_cs) < 2)) stop("a batch group still has fewer than 2 samples; raise or lower MIN_BATCH, or remove that sample in notebook 03")

## PCA before correction
`filterByExpr` keeps genes with enough counts to be informative; `cpm(log = TRUE)` removes differences in sequencing depth; `prcomp` computes the principal components. Two panels: the same map coloured by batch, then by diagnosis. If the batch colours separate and the diagnosis colours don't, the samples group by facility.

In [ ]:
expressed <- filterByExpr(DGEList(counts), group = samples$diagnosis)
pca_of <- function(M) prcomp(t(cpm(DGEList(M[expressed, ]), log = TRUE)))
pca_plot <- function(p, title) {
  share <- round(100 * p$sdev^2 / sum(p$sdev^2), 1)
  par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
  for (by in c("batch", "diagnosis")) {
    f <- factor(samples[[by]]); col <- setNames(hcl.colors(nlevels(f), "Dark 3"), levels(f))
    plot(p$x[, 1], p$x[, 2], col = col[as.character(f)], pch = 19,
         xlab = sprintf("PC1 (%.1f%%)", share[1]), ylab = sprintf("PC2 (%.1f%%)", share[2]),
         main = sprintf("%s, coloured by %s", title, by), cex.main = 0.9)
    legend("topright", legend = levels(f), col = col, pch = 19, cex = 0.6, bty = "n")
  }
}
c(genes = nrow(counts), expressed = sum(expressed))
options(repr.plot.width = 14, repr.plot.height = 6)
p_before <- pca_of(counts)
pca_plot(p_before, "Raw counts")

## ComBat-seq
Fits a negative-binomial model per gene, estimates how each batch shifts the gene's mean and spread, and returns adjusted whole-number counts as if all samples had been sequenced together (Zhang, Parmigiani, Johnson. *NAR Genom Bioinform* 2020). `group = diagnosis` tells it which differences are real and must be kept.

In [ ]:
counts_cs <- tryCatch(
  ComBat_seq(counts, batch = samples$batch_cs, group = samples$diagnosis),
  error = function(e) {
    message("ComBat-seq could not protect diagnosis (", conditionMessage(e), "). Re-running without it: ",
            "any diagnosis difference that lines up with a batch will be partly removed.")
    ComBat_seq(counts, batch = samples$batch_cs)
  })
dimnames(counts_cs) <- dimnames(counts)
c(genes = nrow(counts_cs), samples = ncol(counts_cs), negative = sum(counts_cs < 0), missing = sum(is.na(counts_cs)))

## PCA after correction
The same two panels on the corrected counts. The batch colours should now overlap.

In [ ]:
p_after <- pca_of(counts_cs)
pca_plot(p_after, "ComBat-seq counts")

## How much is batch, how much is diagnosis?
For PC1 and PC2: the fraction of that axis explained by batch, and by diagnosis (R² of a one-factor fit). Before correction batch should dominate; after, batch should drop while diagnosis stays.

In [ ]:
r2 <- function(p, f, k = 2) sapply(seq_len(k), function(i) round(summary(lm(p$x[, i] ~ factor(f)))$r.squared, 2))
ve <- rbind(
  data.frame(stage = "before", factor = "batch",     PC1 = r2(p_before, samples$batch)[1],     PC2 = r2(p_before, samples$batch)[2]),
  data.frame(stage = "before", factor = "diagnosis", PC1 = r2(p_before, samples$diagnosis)[1], PC2 = r2(p_before, samples$diagnosis)[2]),
  data.frame(stage = "after",  factor = "batch",     PC1 = r2(p_after, samples$batch)[1],      PC2 = r2(p_after, samples$batch)[2]),
  data.frame(stage = "after",  factor = "diagnosis", PC1 = r2(p_after, samples$diagnosis)[1],  PC2 = r2(p_after, samples$diagnosis)[2]))
ve

## Save (all CSV)

In [ ]:
write_counts <- function(M, path) {
  write.csv(data.frame(Gene_ID = rownames(M), Gene_Name = gene_name, M, check.names = FALSE), path, row.names = FALSE)
}
one <- samples$one_per_patient == "True"
write_counts(counts_cs,        file.path(OUT_DIR, "counts_combatseq_eligible.csv"))
write_counts(counts_cs[, one], file.path(OUT_DIR, "counts_combatseq_one_per_patient.csv"))
pc_table <- function(p) data.frame(sample_id = samples$sample_id, batch = samples$batch, diagnosis = samples$diagnosis,
                                   p$x[, 1:min(5, ncol(p$x))])
write.csv(pc_table(p_before), file.path(OUT_DIR, "pca_before.csv"), row.names = FALSE)
write.csv(pc_table(p_after),  file.path(OUT_DIR, "pca_after.csv"),  row.names = FALSE)
write.csv(ve, file.path(OUT_DIR, "variance_explained_by_batch_and_diagnosis.csv"), row.names = FALSE)
c(eligible_samples = ncol(counts_cs), one_per_patient = sum(one))
list.files(OUT_DIR)

---
## References
Zhang Y, Parmigiani G, Johnson WE. ComBat-seq: batch effect adjustment for RNA-seq count data. *NAR Genom Bioinform* 2020;2:lqaa078.
Robinson MD, McCarthy DJ, Smyth GK. edgeR. *Bioinformatics* 2010;26:139-140.
Leek JT et al. The sva package for removing batch effects and other unwanted variation. *Bioinformatics* 2012;28:882-883.
Chen Y, Lun ATL, Smyth GK. From reads to genes to pathways. *F1000Research* 2016;5:1438.